In [1]:
import numpy as np

N_samples = 50000
T_burn = 1000
T_avg = 100
L_min = 5
L_max = 80
Denom_max = 11
seed = 0
SAVE_SAMPLES = True

RULES = ['11R', '19R', '24R', '17R', '20', '28', '42', '56', '57', '61', '93']
MAXPLUS_RULES = ['17R-1', '17R-2', '93-1', '93-2']


In [2]:
def flux_table(rule):
    if rule == '11R':
        return np.array([1, 1, 1, 1, 0, 1, 0, 1, 0, 0, 1, 1, -1, 0, 0, 0])
    if rule == '19R':
        return np.array([1, 1, 1, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0])
    if rule == '24R':
        return np.array([1, 1, 1, 1, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0])
    if rule == '17R':
        return np.array([1, 1, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0])
    if rule == '20':
        return np.array([0, 1, 1, 1, 0, 1, 1, 1, 0, 0, 1, 1, 0, 0, 0, 0])
    if rule == '28':
        return np.array([0, 1, 1, 1, 0, 1, 0, 1, 0, 0, 1, 1, -1, 0, 0, 0])
    if rule == '42':
        return np.array([0, 1, 1, 1, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0])
    if rule == '56':
        return np.array([0, 1, 1, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0])
    if rule == '57':
        return np.array([0, 1, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0])
    if rule == '61':
        return np.array([0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0])
    if rule == '93':
        return np.array([0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0])


In [3]:
def flux(u, table, left2, left1, right1):
    a = u[left2]
    b = u[left1]
    c = u
    d = u[right1]
    index = 15 - (8*a + 4*b + 2*c + d)
    return table[index]


In [4]:
def second_count(u, rule, right1, right2, right3):
    L = len(u)
    n = len(u[0])

    if rule == '57':
        count = np.zeros(n, dtype=int)
        length = np.zeros(n, dtype=int)
        for j in range(2*L):
            cell = u[j % L]
            if j >= L:
                count = count + (cell == 0)*(length % 2 == 1)
            length = (length + 1)*cell
        ones = np.zeros(n, dtype=int)
        for j in range(L):
            ones = ones + u[j]
        if L % 2 == 1:
            count = count + (ones == L)
        return count

    if rule == '11R' or rule == '28':
        match = (u == 1)*(u[right2] == 0)
    if rule == '19R' or rule == '56':
        match = (u == 0)*(u[right1] == 1)*(u[right3] == 1)
    if rule == '24R' or rule == '42':
        match = (u == 0)*(u[right2] == 0)*(u[right3] == 1)
    if rule == '17R':
        match = (u == 0)*(u[right1] == 1)*(u[right2] == 1)
    if rule == '20':
        match = (u == 1)*(u[right2] == 1)*(u[right3] == 0)
    if rule == '61' or rule == '93':
        match = (u == 0)*(u[right1] == 1)*(u[right3] == 0)

    count = np.zeros(n, dtype=int)
    for j in range(L):
        count = count + match[j]
    return count


In [5]:
def fundamental_diagram(rule, x, y):
    if rule == '11R':
        return min(2*x, 1-2*y)
    if rule == '19R':
        Q1 = min(2*x, 1-2*x)
        return max(Q1, x-y)
    if rule == '24R':
        Q1 = min(2*x, 1-x-y)
        return max(Q1, 2*x-1)
    if rule == '17R':
        return max(2*y, 2*x-1)
    if rule == '20':
        return min(2*x, 1-x+y)
    if rule == '28':
        return min(2*x, 1-2*y, 2-2*x)
    if rule == '42':
        Q1 = min(2*x, 1-x-y)
        Q2 = max(Q1, 2*x-1)
        return min(Q2, 2-2*x)
    if rule == '56':
        return min(x-y, 2-2*x)
    if rule == '57':
        return min(2*(1-x), x-y)
    if rule == '61':
        return min(2-2*x-y, x-y)
    if rule == '93':
        return min(x-y, 1-x)


In [6]:
results = []
summary_file = open('pca5_table3_errors.csv', 'w')
summary_file.write('Rule,E_max,E_max_below_1e-14\n')

for rule in RULES:
    np.random.seed(seed)
    sizes = np.random.randint(L_min, L_max + 1, N_samples)
    table = flux_table(rule)
    E_max = 0.0
    sample_id = 0

    if SAVE_SAMPLES:
        sample_file = open('pca5_' + rule + '_samples.csv', 'w')
        sample_file.write('Sample,K,Rho1,Rho2,Q,F,Absolute_error\n')

    for L in range(L_min, L_max + 1):
        n = 0
        for s in range(N_samples):
            if sizes[s] == L:
                n = n + 1

        if n > 0:
            p = np.random.rand(n)
            random_values = np.random.rand(n, L)
            u = np.zeros((L, n), dtype=int)
            for j in range(L):
                for s in range(n):
                    if random_values[s, j] < p[s]:
                        u[j, s] = 1

            left2 = []
            left1 = []
            right1 = []
            right2 = []
            right3 = []
            for j in range(L):
                left2.append((j-2) % L)
                left1.append((j-1) % L)
                right1.append((j+1) % L)
                right2.append((j+2) % L)
                right3.append((j+3) % L)

            rho1 = np.zeros(n)
            for j in range(L):
                rho1 = rho1 + u[j]
            rho1 = rho1 / L

            if rule == '57':
                odd_count = second_count(u, rule, right1, right2, right3)

            for t in range(T_burn):
                q = flux(u, table, left2, left1, right1)
                u = u + q - q[right1]

            Q_total = np.zeros(n)
            rho2_total = np.zeros(n)
            for t in range(T_avg):
                q = flux(u, table, left2, left1, right1)
                flow = np.zeros(n)
                for j in range(L):
                    flow = flow + q[j]
                Q_total = Q_total + flow / L

                if rule == '57':
                    count = odd_count
                else:
                    count = second_count(u, rule, right1, right2, right3)
                rho2_total = rho2_total + count / L
                u = u + q - q[right1]

            Q = Q_total / T_avg
            rho2 = rho2_total / T_avg
            for s in range(n):
                sample_id = sample_id + 1
                Q_formula = fundamental_diagram(rule, rho1[s], rho2[s])
                error = abs(Q[s] - Q_formula)
                if error > E_max:
                    E_max = error
                if SAVE_SAMPLES:
                    row = str(sample_id) + ',' + str(L)
                    row = row + ',' + str(rho1[s]) + ',' + str(rho2[s])
                    row = row + ',' + str(Q[s]) + ',' + str(Q_formula)
                    sample_file.write(row + ',' + str(error) + '\n')

    if SAVE_SAMPLES:
        sample_file.close()
    results.append([rule, E_max])
    summary_file.write(rule + ',' + str(E_max) + ',' + str(E_max < 1e-14) + '\n')
    print('PCA5-' + rule, 'E_max =', E_max, 'below 1e-14 =', E_max < 1e-14)

summary_file.close()


PCA5-11R E_max = 2.4424906541753444e-15 below 1e-14 = True
PCA5-19R E_max = 2.4424906541753444e-15 below 1e-14 = True
PCA5-24R E_max = 2.4424906541753444e-15 below 1e-14 = True
PCA5-17R E_max = 2.4424906541753444e-15 below 1e-14 = True
PCA5-20 E_max = 2.3314683517128287e-15 below 1e-14 = True
PCA5-28 E_max = 2.4251434194155763e-15 below 1e-14 = True
PCA5-42 E_max = 1.7763568394002505e-15 below 1e-14 = True
PCA5-56 E_max = 1.7208456881689926e-15 below 1e-14 = True
PCA5-57 E_max = 1.7208456881689926e-15 below 1e-14 = True
PCA5-61 E_max = 1.7763568394002505e-15 below 1e-14 = True
PCA5-93 E_max = 1.7763568394002505e-15 below 1e-14 = True


In [7]:
def minimum(a, b):
    return (a <= b)*a + (a > b)*b

def maximum(a, b):
    return (a >= b)*a + (a < b)*b


In [8]:
def maxplus_flux(u, rule, unit, left2, left1, right1):
    a = u[left2]
    b = u[left1]
    c = u
    d = u[right1]
    if rule == '17R-1':
        return maximum(0, maximum(a+b-unit, b+c-a-d-unit))
    if rule == '17R-2':
        first = minimum(maximum(0, a+b-unit), unit-c)
        second = minimum(maximum(0, b+c-unit), unit-d)
        return maximum(a+b-unit, first+second)
    if rule == '93-1':
        return minimum(a+d, minimum(b, unit-c))
    if rule == '93-2':
        return minimum(maximum(a, d), minimum(b, unit-c))


In [9]:
def maxplus_second_sum(u, rule, unit, left1, right1, right2):
    a = u[left1]
    b = u
    c = u[right1]
    d = u[right2]
    if rule == '17R-1':
        r = maximum(0, a+b-c-unit)
    if rule == '17R-2':
        r = minimum(maximum(0, a+b-unit), unit-c)
    if rule == '93-1':
        r = maximum(0, b-a-d)
    if rule == '93-2':
        r = maximum(0, b-maximum(a, d))
    count = np.zeros(len(u[0]), dtype=int)
    for j in range(len(u)):
        count = count + r[j]
    return count


In [10]:
def maxplus_formula(rule, x, y):
    if rule == '17R-1' or rule == '17R-2':
        return max(2*y, 2*x-1)
    if rule == '93-1' or rule == '93-2':
        return min(x-y, 1-x)


In [11]:
maxplus_results = []
summary_file = open('maxplus_errors.csv', 'w')
summary_file.write('Rule,E_max,E_max_below_1e-14,Maximum_integer_residual\n')

for rule in MAXPLUS_RULES:
    np.random.seed(seed)
    sizes = np.random.randint(L_min, L_max + 1, N_samples)
    E_max = 0.0
    integer_residual_max = 0
    sample_id = 0

    if SAVE_SAMPLES:
        sample_file = open('mpca5_' + rule + '_samples.csv', 'w')
        sample_file.write('Sample,K,Denominator,Rho1,Rho2,Q,F,Absolute_error\n')

    for L in range(L_min, L_max + 1):
        n = 0
        for s in range(N_samples):
            if sizes[s] == L:
                n = n + 1

        if n > 0:
            unit = np.random.randint(1, Denom_max + 1, n)
            u = np.zeros((L, n), dtype=int)
            for s in range(n):
                values = np.random.randint(0, unit[s] + 1, L)
                for j in range(L):
                    u[j, s] = values[j]

            left2 = []
            left1 = []
            right1 = []
            right2 = []
            for j in range(L):
                left2.append((j-2) % L)
                left1.append((j-1) % L)
                right1.append((j+1) % L)
                right2.append((j+2) % L)

            particles = np.zeros(n, dtype=int)
            for j in range(L):
                particles = particles + u[j]
            rho1 = particles / (unit*L)

            for t in range(T_burn):
                q = maxplus_flux(u, rule, unit, left2, left1, right1)
                u = u + q - q[right1]

            Q_total = np.zeros(n)
            rho2_total = np.zeros(n)
            total_flux = np.zeros(n, dtype=int)
            total_second = np.zeros(n, dtype=int)

            for t in range(T_avg):
                q = maxplus_flux(u, rule, unit, left2, left1, right1)
                flow = np.zeros(n, dtype=int)
                for j in range(L):
                    flow = flow + q[j]
                second = maxplus_second_sum(u, rule, unit, left1, right1, right2)
                Q_total = Q_total + flow / (unit*L)
                rho2_total = rho2_total + second / (unit*L)
                total_flux = total_flux + flow
                total_second = total_second + second
                u = u + q - q[right1]

            Q = Q_total / T_avg
            rho2 = rho2_total / T_avg
            for s in range(n):
                sample_id = sample_id + 1
                predicted = maxplus_formula(rule, rho1[s], rho2[s])
                error = abs(Q[s] - predicted)
                if error > E_max:
                    E_max = error

                denominator = unit[s]*L*T_avg
                particle_total = particles[s]*T_avg
                if rule == '17R-1' or rule == '17R-2':
                    predicted_total = max(2*total_second[s], 2*particle_total-denominator)
                if rule == '93-1' or rule == '93-2':
                    predicted_total = min(particle_total-total_second[s], denominator-particle_total)
                residual = abs(total_flux[s] - predicted_total)
                if residual > integer_residual_max:
                    integer_residual_max = residual

                if SAVE_SAMPLES:
                    row = str(sample_id) + ',' + str(L) + ',' + str(unit[s])
                    row = row + ',' + str(rho1[s]) + ',' + str(rho2[s])
                    row = row + ',' + str(Q[s]) + ',' + str(predicted)
                    sample_file.write(row + ',' + str(error) + '\n')

    if SAVE_SAMPLES:
        sample_file.close()
    maxplus_results.append([rule, E_max, integer_residual_max])
    row = rule + ',' + str(E_max) + ',' + str(E_max < 1e-14)
    summary_file.write(row + ',' + str(integer_residual_max) + '\n')
    print('mPCA5-' + rule, 'E_max =', E_max, 'integer residual =', integer_residual_max)

summary_file.close()


mPCA5-17R-1 E_max = 1.5543122344752192e-15 integer residual = 0
mPCA5-17R-2 E_max = 1.5543122344752192e-15 integer residual = 0
mPCA5-93-1 E_max = 1.6653345369377348e-15 integer residual = 0
mPCA5-93-2 E_max = 1.6653345369377348e-15 integer residual = 0


In [12]:
u = np.array([[2, 3], [2, 3], [2, 0], [2, 3], [2, 3], [2, 0]])
unit = np.array([3, 3])
L = 6

left2 = []
left1 = []
right1 = []
right2 = []
for j in range(L):
    left2.append((j-2) % L)
    left1.append((j-1) % L)
    right1.append((j+1) % L)
    right2.append((j+2) % L)

particles = np.zeros(2, dtype=int)
for j in range(L):
    particles = particles + u[j]
rho1 = particles / (unit*L)

for t in range(T_burn):
    q = maxplus_flux(u, '17R-1', unit, left2, left1, right1)
    u = u + q - q[right1]

total_flux = np.zeros(2, dtype=int)
total_second = np.zeros(2, dtype=int)
for t in range(T_avg):
    q = maxplus_flux(u, '17R-1', unit, left2, left1, right1)
    for j in range(L):
        total_flux = total_flux + q[j]
    second = maxplus_second_sum(u, '17R-2', unit, left1, right1, right2)
    total_second = total_second + second
    u = u + q - q[right1]

Q = total_flux / (unit*L*T_avg)
rho2 = total_second / (unit*L*T_avg)
names = ['Uniform_two_thirds', '110110']
output_file = open('swapped_density_check.csv', 'w')
output_file.write('Configuration,Rho1,Swapped_rho2,Q\n')
for s in range(2):
    print(names[s], 'rho1 =', rho1[s], 'swapped rho2 =', rho2[s], 'Q =', Q[s])
    row = names[s] + ',' + str(rho1[s]) + ',' + str(rho2[s])
    output_file.write(row + ',' + str(Q[s]) + '\n')
output_file.close()
print('Same densities =', rho1[0] == rho1[1] and rho2[0] == rho2[1])
print('Different flows =', Q[0] != Q[1])


Uniform_two_thirds rho1 = 0.6666666666666666 swapped rho2 = 0.3333333333333333 Q = 0.3333333333333333
110110 rho1 = 0.6666666666666666 swapped rho2 = 0.3333333333333333 Q = 0.6666666666666666
Same densities = True
Different flows = True
